<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/lessons/lesson_31_redis_overview/note_lesson_31_redis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Урок 31 — Redis: лічильники, кеш, черги й рейтинг диспетчерської

База PostgreSQL з уроку 30 — **джерело правди** «Смачно + Таксі»: замовлення, кур'єри, гроші. Але є дані, для яких вона не найкращий інструмент: лічильник «замовлень за хвилину», що оновлюється сотні разів на секунду; звіт, який рахується 2 секунди, а відкривають його десять разів на хвилину; SMS, які треба відправити **у фоні**; рейтинг кур'єрів, що має змінюватися миттєво. Для цього поруч із PostgreSQL ставлять **Redis** — сервер «ключ → значення» в оперативній пам'яті.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | запускаємо Redis і підключаємось з Python | 0.1 | |
| 1 | ключі й рядки: `SET`, `GET`, `DEL`, `EXISTS`, `SET NX` | 1.1–1.4 | |
| 2 | `INCR`: атомарний лічильник проти «прочитати-додати-записати» з потоків | 2.1–2.4 | |
| 3 | час життя: `EXPIRE`, `TTL`, ключ, що зникає на очах | 3.1–3.5 | |
| 4 | **list**: черга, стек, історія — і `Queue` / `Stack` з уроку 29 | 4.1–4.5 | |
| 5 | **hash**: картка кур'єра — і `dict` | 5.1–5.4 | |
| 6 | **set**: унікальні клієнти — і `set` | 6.1–6.3 | |
| 7 | **sorted set**: рейтинг і черга за дедлайном — і `MinHeap` | 7.1–7.4 | **Задача 1** |
| 8 | redis-py і `pipeline`: заміряємо час | 8.1–8.4 | |
| 9 | кеш **cache-aside**: hit / miss, застарілі дані, інвалідація | 9.1–9.3 | **Задача 2** |
| 10 | черга задач: producer і consumer, `BLPOP` | 10.1–10.3 | |
| 11 | Pub/Sub: повідомлення всім, хто слухає | 11.1–11.3 | |
| 12 | персистентність RDB / AOF, `maxmemory-policy` через `CONFIG GET` | 12.1–12.4 | |
| 13 | коли Redis не потрібен: міряємо | 13.1 | |
| 14 | протокол RESP через звичайний сокет | 14.1–14.2 | |
| 15 | практика: rate limit | 15.1–15.3 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, константи) лише створює функцію — **запуск завжди в наступній клітинці**. Так приклад можна перезапускати, не перевизначаючи функції.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Час у тебе буде іншим, ніж у тексті, — важливе **співвідношення** («у 10 разів швидше»), а не мілісекунди.
- Задачі — три, у кожної перевірка в наступній клітинці.
- Теорія й схеми — у книзі: [Урок 31](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/lesson_31/).

## 🗺️ Загальна картина: де Redis серед баз даних

Перш ніж вчити команди, подивимось, **яке місце Redis займає в системі**. У «Смачно + Таксі» працюють три сховища, і в кожного своя роль:

```
          клієнт натиснув «Замовити»
                     │
                     ▼
 ┌────────────────────────────────────────────┐
 │ ⚡ Redis — гаряча пам'ять (ЦЕЙ УРОК)        │  щосекунди: лічильники, кеш меню,
 │  order:42 (HASH) · orders:new (черга)      │  черга замовлень, де зараз кур'єри,
 │  couriers:geo · couriers:free · lb:*       │  рейтинг. Швидко, але тимчасово.
 └────────────────────────────────────────────┘
          │ замовлення доставлено     ▲ на старті: ресторани, кур'єри
          ▼                           │
 ┌────────────────────────────────────────────┐
 │ 🐘 PostgreSQL — джерело правди (урок 30)    │  restaurants · menu_items ·
 │  те, що не можна втратити: гроші, історія  │  couriers · orders (доставлені)
 └────────────────────────────────────────────┘
          │ вивантаження в parquet
          ▼
 ┌────────────────────────────────────────────┐
 │ 🦆 DuckDB — аналітика (бонус)               │  «середній час доставки за рік»
 └────────────────────────────────────────────┘
```

**Правило вибору:** не можна втратити → PostgreSQL; потрібно «зараз», швидко або кільком програмам разом → Redis; питання до всієї історії → DuckDB.

Кожен розділ уроку — одна задача з цієї схеми: лічильник (розд. 2–3), кеш (розд. 9), черга (розд. 4 і 10), рейтинг, «хто поруч».

- 🧭 **Інтерактивний гайд** [«Бази даних: загальна картина»](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/bonus/%D0%91%D0%B0%D0%B7%D0%B8%20%D0%B4%D0%B0%D0%BD%D0%B8%D1%85%20%E2%80%94%20%D0%B7%D0%B0%D0%B3%D0%B0%D0%BB%D1%8C%D0%BD%D0%B0%20%D0%BA%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D0%B0.html) — завантаж HTML і відкрий у браузері. У ньому: картки сутностей, пісочниця Redis, анімації патернів, схема проєкту з логами, опитувальник «куди покласти дані» і конструктор своєї бази.
- 🍕 **Проєкт, де все працює разом:** [`taxi_delivery_redis`](https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_3/lessons/taxi_delivery_redis/) — почни з `ARCHITECTURE.md`.

---
## 0. Redis і підключення

- **Google Colab:** клітинка нижче встановить Redis у віртуальну машину Colab і запустить сервер. Сервер живе, доки працює сесія.
- **Свій комп'ютер:** запусти Redis заздалегідь: `docker run --name smachno-redis -p 6379:6379 -d redis:7` або `sudo apt install redis-server`. Інша адреса чи порт — задай змінну середовища `REDIS_URL` перед запуском Jupyter, наприклад `REDIS_URL=redis://localhost:6390/15`.

Ноутбук працює в **окремій базі Redis з номером 15** (останнє число в адресі), а всі його ключі починаються з `l31:` — це «простір імен» уроку. Прибираємо **лише** ключі `l31:*`, тож твої інші дані в Redis не постраждають, а ноутбук можна запускати скільки завгодно разів.

In [1]:
import os
import subprocess
import sys
import time

if "google.colab" in sys.modules:
    subprocess.run("apt-get -qq update && apt-get -qq install -y redis-server > /dev/null", shell=True, check=True)
    subprocess.run("redis-server --daemonize yes", shell=True, check=True)
    time.sleep(1)                     # даємо серверу секунду на старт

try:
    import redis                      # клієнт уже встановлено
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "redis"], check=True)
print("готово")

готово


In [39]:
import json
import socket
import threading
from collections import deque
import heapq

import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/15")
PREFIX = "l31:"                       # усі ключі уроку починаються з цього

r = redis.Redis.from_url(REDIS_URL, decode_responses=True)


def cleanup():
    """Видалити всі ключі уроку (і тільки їх). Повертає, скільки видалено."""
    keys = list(r.scan_iter(PREFIX + "*"))
    return r.delete(*keys) if keys else 0

### Приклад 0.1. Перший запит

`ping()` — «ти живий?»: `True`, якщо сервер відповів. `info("server")` — словник з відомостями про сервер; візьмемо з нього версію. `cleanup()` прибирає ключі попереднього запуску.

In [40]:
print("зв'язок:", r.ping())
print("версія Redis:", r.info("server")["redis_version"])
print("прибрано старих ключів уроку:", cleanup())

зв'язок: True
версія Redis: 8.12.0
прибрано старих ключів уроку: 8


Якщо замість `True` бачиш `ConnectionError` — сервер не запущено або адреса інша (див. `REDIS_URL` вище).

```
 сервіс диспетчерської ──лічильники, кеш, черги──▶  Redis       (пам'ять, порт 6379)
        (Python)       ──замовлення, гроші──────▶  PostgreSQL  (диск, порт 5432)
 воркер SMS ──────────── бере задачі з черги ───▶  Redis
```

Правило: усе, що не можна втратити, — у PostgreSQL. Redis тримає те, що можна **відновити** (кеш) або що **живе недовго** (лічильник за хвилину, задача в черзі).

---
## 1. Ключі й рядки

Redis — це великий словник на сервері. Найпростіше значення — рядок: `SET` записує, `GET` читає, `DEL` видаляє, `EXISTS` перевіряє. У redis-py кожна команда — метод з тією самою назвою в нижньому регістрі.

### Приклад 1.1. `SET`, `GET`, `EXISTS`, `DEL`

**Задача:** записати ім'я кур'єра й прочитати його назад. **Зміна:** спробуємо прочитати ключ, якого немає, і порахуємо, скільки з двох ключів існує.

In [13]:
print("SET:", r.set("l31:courier:1:name", "Оксана"))

SET: True


In [14]:
print("GET:", r.get("l31:courier:1:name"))

GET: Оксана


In [15]:
print("GET неіснуючого:", r.get("l31:courier:99:name"))

GET неіснуючого: None


In [16]:
print("EXISTS двох ключів:", r.exists("l31:courier:1:name", "l31:courier:99:name"))

EXISTS двох ключів: 1


In [17]:
print("DEL:", r.delete("l31:courier:1:name", "l31:courier:99:name"))
print("після DEL:", r.get("l31:courier:1:name"))

DEL: 1
після DEL: None


- `SET` повертає `True` (у `redis-cli` — `OK`);
- `GET` неіснуючого ключа — `None` (у `redis-cli` — `nil`);
- `EXISTS` повертає **скільки** з переданих ключів існує, а `DEL` — скільки ключів справді видалив: другого ключа не було, тому 1, а не 2.

### Приклад 1.2. Двокрапки — домовленість, а не синтаксис

**Задача:** записати картки трьох кур'єрів і знайти всі ключі кур'єрів. Назва ключа — просто рядок; двокрапки ділять її на «простори імен»: `l31:courier:2:name` — «урок 31, кур'єр 2, ім'я». `scan_iter` обходить ключі за шаблоном потроху, не блокуючи сервер (команда `KEYS` на великій базі зупинила б Redis на секунди).

In [18]:
for courier_id, name in [(1, "Оксана"), (2, "Тарас"), (3, "Ігор")]:
    r.set(f"l31:courier:{courier_id}:name", name)
r.set("l31:promo:SUMMER10", "active")

True

In [19]:
print(sorted(r.scan_iter("l31:courier:*")))


['l31:courier:1:name', 'l31:courier:2:name', 'l31:courier:3:name']


In [20]:
print("тип значення:", r.type("l31:courier:1:name"))

тип значення: string


Шаблон `l31:courier:*` знайшов три ключі кур'єрів і не зачепив промокод. `TYPE` показує тип значення — зараз `string`; далі побачимо `list`, `hash`, `set`, `zset`.

### Приклад 1.3. Redis зберігає байти

**🔮 Прогноз:** записуємо число `7`. Що поверне `GET` — `7` чи `'7'`? А що поверне клієнт **без** `decode_responses=True` для імені «Оксана»?

<details>
<summary>Відповідь</summary>

`'7'` — рядок: Redis не знає про типи Python. Без `decode_responses` відповіді приходять як `bytes`: `b'\xd0\x9e\xd0\xba...'` — UTF-8-коди кириличних літер.

</details>

In [21]:
raw = redis.Redis.from_url(REDIS_URL)          # без decode_responses

r.set("l31:orders:today", 7)
print(repr(r.get("l31:orders:today")), "→ int:", int(r.get("l31:orders:today")) + 1)
print(repr(raw.get("l31:courier:1:name")))
print(raw.get("l31:courier:1:name").decode("utf-8"))
raw.close()

'7' → int: 8
b'\xd0\x9e\xd0\xba\xd1\x81\xd0\xb0\xd0\xbd\xd0\xb0'
Оксана


Числа повертаються рядками — перетворюй через `int()` чи `float()`. Без `decode_responses=True` кожну відповідь довелося б декодувати вручну (`.decode("utf-8")`), тому в уроці клієнт `r` створено з ним.

### Приклад 1.4. `SET NX`: записати, лише якщо ключа ще немає

**Задача:** двоє диспетчерів одночасно натискають «взяти замовлення #104». Виграти має **перший**. `nx=True` — «записати тільки якщо ключа немає»: перевірка й запис — одна команда.

In [22]:
r.delete("l31:order:104:owner")

0

In [23]:

print("Марина:", r.set("l31:order:104:owner", "Марина", nx=True))
print("Олег:  ", r.set("l31:order:104:owner", "Олег", nx=True))
print("власник:", r.get("l31:order:104:owner"))

Марина: True
Олег:   None
власник: Марина


Перший `SET NX` повернув `True`, другий — `None`: ключ уже був, і значення не змінилось. Якби ми зробили `if not r.exists(...)` і потім `r.set(...)`, між двома запитами міг би вклинитися інший диспетчер — це та сама гонитва, що в наступному розділі.

---
## 2. `INCR`: лічильник однією командою

### Приклад 2.1. `INCR`, `INCRBY`, `DECR`, `INCRBYFLOAT`

**Задача:** лічильник замовлень за день. `INCR` створює ключ зі значенням 0, якщо його не було, і додає 1 — і **повертає нове значення**. `INCRBY` додає довільне ціле, `DECR` віднімає 1, `INCRBYFLOAT` — дробове.

In [24]:
r.delete("l31:orders:today", "l31:revenue:today")

1

In [25]:

print(r.incr("l31:orders:today"))
print(r.incr("l31:orders:today"))
print(r.incrby("l31:orders:today", 5))
print(r.decr("l31:orders:today"))
print(r.incrbyfloat("l31:revenue:today", 245.5))
print(r.incrbyfloat("l31:revenue:today", 120.25))

1
2
7
6
245.5
365.75


Ключа не було — перший `INCR` дав 1. Далі 2, 2 + 5 = 7, 7 − 1 = 6. Гроші — `INCRBYFLOAT`: 245.5 + 120.25 = 365.75.

А якщо в ключі не число? Redis відмовить помилкою — і нічого не змінить:

In [26]:
try:
    r.incr("l31:courier:1:name")
except redis.ResponseError as error:
    print("помилка Redis:", error)
print(r.get("l31:courier:1:name"))

помилка Redis: value is not an integer or out of range
Оксана


### Приклад 2.2. Гонитва покроково

Чому не можна рахувати «прочитати, додати 1, записати»? Розіграємо дві каси вручну, у тому порядку, в якому їхні запити можуть прийти до Redis:

```
каса 1: GET → 7
каса 2: GET → 7          ← прочитала ДО того, як каса 1 записала
каса 1: SET 8
каса 2: SET 8            ← одне замовлення втрачено
```

In [27]:
r.set("l31:orders:today", 7)




print("два замовлення, а лічильник:", r.get("l31:orders:today"))

два замовлення, а лічильник: 8


In [28]:
cashier_1 = int(r.get("l31:orders:today"))     # каса 1 читає
cashier_2 = int(r.get("l31:orders:today"))     # каса 2 читає те саме

In [29]:
r.set("l31:orders:today", cashier_1 + 1)       # каса 1 пише
r.set("l31:orders:today", cashier_2 + 1)       # каса 2 перезаписує

True

In [30]:
print("два замовлення, а лічильник:", r.get("l31:orders:today"))


два замовлення, а лічильник: 9


Два замовлення, а лічильник виріс на 1 — це стан гонитви з уроку 28. Тепер те саме з `INCR`: Redis виконує команди **по одній**, і «прочитати-додати-записати» всередині `INCR` не може перерватися.

In [31]:
r.set("l31:orders:today", 7)
print("каса 1:", r.incr("l31:orders:today"))
print("каса 2:", r.incr("l31:orders:today"))

каса 1: 8
каса 2: 9


### Приклад 2.3. Чотири «сервери» рахують через `GET` + `SET`

Вручну ми самі обрали поганий порядок. Чи трапиться він насправді? Чотири потоки — чотири сервери — приймають по 500 замовлень і рахують їх «по-простому».

**🔮 Прогноз:** скільки буде в лічильнику — 2000, трохи менше, чи значно менше?

<details>
<summary>Відповідь</summary>

Значно менше, і щоразу інше число. Кожен `GET` і `SET` — окремий запит мережею; поки один потік чекає відповіді, інші встигають прочитати те саме старе значення.

</details>

In [33]:
def count_order_buggy():
    current = int(r.get("l31:orders:buggy") or 0)
    r.set("l31:orders:buggy", current + 1)


def run_servers(count_order, servers=4, orders_each=500):
    """Запустити `servers` потоків, кожен викликає count_order() `orders_each` разів."""
    def server():
        for _ in range(orders_each):
            count_order()

    threads = [threading.Thread(target=server) for _ in range(servers)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()

In [34]:
r.delete("l31:orders:buggy")
run_servers(count_order_buggy)
counted = int(r.get("l31:orders:buggy"))
print(f"очікували 2000, маємо: {counted} — втрачено {2000 - counted}")

очікували 2000, маємо: 656 — втрачено 1344


Частина замовлень зникла (у тебе число буде іншим). `threading.Lock` з уроку 28 тут не врятує: справжні сервери — різні процеси на різних машинах, спільного замка в них немає. Спільний у них лише Redis.

### Приклад 2.4. Те саме з `INCR`

In [35]:
def count_order():
    r.incr("l31:orders:today")

In [36]:
r.delete("l31:orders:today")
run_servers(count_order)
print("очікували 2000, маємо:", r.get("l31:orders:today"))
assert r.get("l31:orders:today") == "2000"

очікували 2000, маємо: 2000


Рівно 2000 — щоразу. Одна атомарна команда замість двох — і гонитви немає. **Правило:** якщо значення змінюється з кількох місць, змінюй його командою Redis (`INCR`, `HINCRBY`, `ZINCRBY`, `SET NX`), а не «прочитав у Python — записав назад».

---
## 3. Час життя: `EXPIRE` і `TTL`

Лічильник «замовлень за цю хвилину» має зникнути сам. Кожному ключу можна задати **час життя** (TTL, time to live), і Redis видалить ключ, коли час мине.

**🔮 Прогноз:** що поверне `r.ttl(key)` для ключа без часу життя? А для ключа, якого немає?

<details>
<summary>Відповідь</summary>

`-1` — ключ живе вічно; `-2` — ключа немає.

</details>

### Приклад 3.1. `EXPIRE`, `TTL`, `SET … EX`

In [ ]:
r.set("l31:promo:SUMMER10", "active")
print("TTL без часу життя:", r.ttl("l31:promo:SUMMER10"))
print("TTL неіснуючого:   ", r.ttl("l31:no:such:key"))

print("EXPIRE:", r.expire("l31:promo:SUMMER10", 60))
print("TTL після EXPIRE:  ", r.ttl("l31:promo:SUMMER10"))

r.set("l31:report:week", "...", ex=300)            # записати й одразу задати час життя
print("TTL після SET EX:  ", r.ttl("l31:report:week"))
print("EXPIRE неіснуючого:", r.expire("l31:no:such:key", 60))

`-1` і `-2` — як у прогнозі. `EXPIRE` повертає `True`, якщо час життя задано, і `False`, якщо ключа немає. `SET … ex=300` — запис і час життя однією командою: між ними ніхто не вклиниться, і ключ не залишиться «вічним», якщо програма впаде між двома командами.

### Приклад 3.2. Ключ зникає на очах

**Задача:** промокод «лише 3 секунди». Будемо щосекунди питати Redis, що з ключем.

In [ ]:
def watch_key(key, seconds):
    """Щосекунди друкувати TTL і значення ключа."""
    for second in range(seconds + 1):
        print(f"{second} с: TTL={r.ttl(key):>2}  GET={r.get(key)!r}  EXISTS={r.exists(key)}")
        time.sleep(1)

In [ ]:
r.set("l31:promo:FLASH", "мінус 20 %", ex=3)
watch_key("l31:promo:FLASH", seconds=4)

TTL зменшується щосекунди, а коли доходить до нуля — `GET` повертає `None`, `EXISTS` — 0, а `TTL` — `-2`: ключа більше немає. Ніхто не викликав `DEL` — Redis прибрав ключ сам. Саме так живуть кеш і тимчасові лічильники.

### Приклад 3.3. Мілісекунди і `PERSIST`

`PEXPIRE` / `PTTL` — те саме в мілісекундах. `PERSIST` знімає час життя: ключ знову «вічний».

In [ ]:
r.set("l31:session:anna", "кошик: 2 піци")
r.pexpire("l31:session:anna", 1500)
print("PTTL:", r.pttl("l31:session:anna"), "мс")
print("TTL: ", r.ttl("l31:session:anna"), "с")
print("PERSIST:", r.persist("l31:session:anna"))
print("TTL після PERSIST:", r.ttl("l31:session:anna"))

`TTL` округлює до секунд, `PTTL` показує точніше. Після `PERSIST` TTL став `-1`.

### Приклад 3.4. Пастка: звичайний `SET` стирає час життя

**🔮 Прогноз:** у ключа TTL 60 секунд. Ми оновлюємо значення звичайним `SET`. Який TTL тепер?

<details>
<summary>Відповідь</summary>

`-1`. `SET` замінює ключ повністю — разом із часом життя. Щоб зберегти TTL, є параметр `keepttl=True`.

</details>

In [ ]:
r.set("l31:courier:1:status", "вільний", ex=60)
r.set("l31:courier:1:status", "у дорозі")
print("після SET:        ", r.ttl("l31:courier:1:status"))

r.set("l31:courier:1:status", "вільний", ex=60)
r.set("l31:courier:1:status", "у дорозі", keepttl=True)
print("після SET KEEPTTL:", r.ttl("l31:courier:1:status"))

Без `keepttl` статус кур'єра став би вічним, і «завислий» статус ніколи б не зник. Зате `INCR` час життя **не** скидає — це використаємо в наступному прикладі.

### Приклад 3.5. Лічильник переглядів на 60 секунд

**Задача:** лічильник переглядів сторінки меню `l31:views:<page>`; кожен виклик додає 1 атомарно й повертає нове значення. Ключ живе **60 секунд від першого перегляду**: TTL ставимо лише тоді, коли лічильник щойно став 1. Якби ставили на кожному перегляді, ключ ніколи б не зник, поки сторінку відкривають.

In [ ]:
def count_view(page):
    key = f"l31:views:{page}"
    count = r.incr(key)
    if count == 1:                 # перший перегляд — запускаємо годинник
        r.expire(key, 60)
    return count

In [ ]:
r.delete("l31:views:pizza", "l31:views:sushi")
print([count_view("pizza") for _ in range(3)])
print(count_view("sushi"))
print("TTL піци:", r.ttl("l31:views:pizza"))

assert [count_view("pizza") for _ in range(2)] == [4, 5]
assert 0 < r.ttl("l31:views:pizza") <= 60

Піца — 1, 2, 3, суші — свій лічильник з 1. TTL — близько 60 секунд і після нових переглядів не скидається на 60, бо `EXPIRE` виконався лише раз.

---
## 4. List: черга, стек, історія

Значенням ключа може бути не лише рядок. Основні типи — старі знайомі з уроку 29:

| Урок 29 | Redis | Команди | Для диспетчерської |
|---|---|---|---|
| `Queue` / `Stack` | **list** | `RPUSH`, `LPUSH`, `LPOP`, `RPOP`, `BLPOP`, `LRANGE`, `LTRIM` | черга SMS, історія дій |
| `dict` | **hash** | `HSET`, `HGET`, `HGETALL`, `HINCRBY` | картка кур'єра |
| `set` | **set** | `SADD`, `SISMEMBER`, `SCARD`, `SINTER` | унікальні клієнти дня |
| `MinHeap` | **sorted set** | `ZADD`, `ZINCRBY`, `ZRANGE`, `ZPOPMIN` | рейтинг, черга за дедлайном |
| `LRUCache` | увесь Redis з `maxmemory-policy allkeys-lru` | налаштування сервера | кеш (розділ 12) |

Головна різниця: структура з уроку 29 живе в пам'яті **твоєї програми** і зникає разом з нею; структура Redis живе на сервері, і з нею одночасно працюють усі програми сервісу.

### Приклад 4.1. Черга FIFO: `RPUSH` + `LPOP` і `Queue` з уроку 29

**Задача:** черга SMS — хто перший прийшов, того першим і обслужили. `Queue` з уроку 29 тримала елементи в `collections.deque`; поруч — те саме в Redis.

In [37]:
SMS = ["Анна: замовлення #101 прийнято", "Богдан: кур'єр виїхав", "Віра: замовлення доставлено"]

In [42]:
local_queue = deque()                      # як Queue з уроку 29
for text in SMS:
    local_queue.append(text)               # enqueue
print("deque  →", local_queue.popleft())   # dequeue

r.delete("l31:sms:queue")
for text in SMS:
    print("RPUSH повернув довжину:", r.rpush("l31:sms:queue", text))
print("LRANGE:", r.lrange("l31:sms:queue", 0, -1))
print("Redis  →", r.lpop("l31:sms:queue"))
print("LLEN:", r.llen("l31:sms:queue"))

deque  → Анна: замовлення #101 прийнято
RPUSH повернув довжину: 1
RPUSH повернув довжину: 2
RPUSH повернув довжину: 3
LRANGE: ['Анна: замовлення #101 прийнято', "Богдан: кур'єр виїхав", 'Віра: замовлення доставлено']
Redis  → Анна: замовлення #101 прийнято
LLEN: 2


`RPUSH` («right push») додає в хвіст і повертає нову довжину списку: 1, 2, 3. `LPOP` бере з голови — найстаріше повідомлення, як `popleft()`. `LRANGE 0 -1` — усі елементи від першого до останнього (`-1` — останній, як у Python).

### Приклад 4.2. Стек: кнопка «Скасувати»

**Задача:** диспетчер призначає кур'єрів і іноді помиляється — «Скасувати» відкочує **останню** дію (LIFO, як `Stack` з уроку 29). У стеку беруть з того самого кінця, куди кладуть: `RPUSH` + `RPOP`.

In [43]:
ACTIONS = ["#101 → Оксана", "#102 → Тарас", "#103 → Ігор"]

In [44]:
local_stack = []                            # як Stack з уроку 29
r.delete("l31:undo")
for action in ACTIONS:
    local_stack.append(action)
    r.rpush("l31:undo", action)

print("list   скасувати:", local_stack.pop(), "| потім:", local_stack.pop())
print("Redis  скасувати:", r.rpop("l31:undo"), "| потім:", r.rpop("l31:undo"))
print("лишилось у Redis:", r.lrange("l31:undo", 0, -1))

list   скасувати: #103 → Ігор | потім: #102 → Тарас
Redis  скасувати: #103 → Ігор | потім: #102 → Тарас
лишилось у Redis: ['#101 → Оксана']


Обидва стеки віддали дії у зворотному порядку: спершу #103, потім #102. `RPOP` з порожнього списку повернув би `None` — як «стек порожній», але без винятку.

### Приклад 4.3. Індекси: `LINDEX`, від'ємні номери, `LRANGE` шматка

In [45]:
r.delete("l31:route")
r.rpush("l31:route", "Поділ", "Оболонь", "Центр", "Печерськ", "Лук'янівка")
print("перший: ", r.lindex("l31:route", 0))
print("останній:", r.lindex("l31:route", -1))
print("2–4:    ", r.lrange("l31:route", 1, 3))
print("поза межами:", r.lindex("l31:route", 10))

перший:  Поділ
останній: Лук'янівка
2–4:     ['Оболонь', 'Центр', 'Печерськ']
поза межами: None


Індекси — як у Python, але **кінець `LRANGE` включно**: `LRANGE 1 3` — три елементи (у Python `route[1:4]`). Індекс поза межами — `None`, а не `IndexError`.

### Приклад 4.4. Історія останніх 5 дій: `LPUSH` + `LTRIM` і `deque(maxlen=5)`

**Задача:** показувати на екрані диспетчера лише 5 останніх подій, щоб список не ріс вічно. У Python — `deque(maxlen=5)`; у Redis — додати на початок і обрізати до 5 елементів.

In [46]:
def log_event(event):
    """Додати подію на початок історії й залишити тільки 5 найсвіжіших."""
    r.lpush("l31:events", event)
    r.ltrim("l31:events", 0, 4)

In [47]:
r.delete("l31:events")
history = deque(maxlen=5)
for number in range(101, 109):
    event = f"замовлення #{number}"
    log_event(event)
    history.appendleft(event)

print("Redis:", r.lrange("l31:events", 0, -1))
print("deque:", list(history))
assert r.lrange("l31:events", 0, -1) == list(history)

Redis: ['замовлення #108', 'замовлення #107', 'замовлення #106', 'замовлення #105', 'замовлення #104']
deque: ['замовлення #108', 'замовлення #107', 'замовлення #106', 'замовлення #105', 'замовлення #104']


Було 8 подій, лишилось 5 найсвіжіших (#108…#104) — і в Redis, і в `deque`. `LTRIM 0 4` залишає елементи з 0 по 4 включно.

### Приклад 4.5. `enqueue_sms` і `next_sms`

Загорнемо чергу у дві функції: `enqueue_sms(text)` кладе текст у хвіст `l31:queue:sms`, `next_sms()` забирає **найстаріше** повідомлення або `None`, якщо черга порожня.

In [48]:
def enqueue_sms(text):
    r.rpush("l31:queue:sms", text)


def next_sms():
    return r.lpop("l31:queue:sms")           # None, якщо черга порожня

In [49]:
r.delete("l31:queue:sms")
for text in ["#101 прийнято", "#102 кур'єр їде", "#103 доставлено"]:
    enqueue_sms(text)
print("у черзі:", r.llen("l31:queue:sms"))
print([next_sms() for _ in range(4)])

assert r.llen("l31:queue:sms") == 0

у черзі: 3
['#101 прийнято', "#102 кур'єр їде", '#103 доставлено', None]


Три повідомлення вийшли в тому порядку, в якому прийшли, а четвертий виклик дав `None` — черга порожня. У розділі 10 цю чергу розберуть окремі воркери.

---
## 5. Hash: картка кур'єра

Hash — словник усередині одного ключа: поле → значення.

### Приклад 5.1. `HSET`, `HGET`, `HGETALL` і `dict`

In [ ]:
card = {"name": "Оксана", "district": "Поділ", "delivered": 3}

In [ ]:
r.delete("l31:courier:1")
print("HSET нових полів:", r.hset("l31:courier:1", mapping=card))
print("HGET district:   ", r.hget("l31:courier:1", "district"))
print("HGETALL:         ", r.hgetall("l31:courier:1"))
print("dict:            ", card)
print("HSET знову:      ", r.hset("l31:courier:1", "district", "Оболонь"))
print("district тепер:  ", r.hget("l31:courier:1", "district"))

Перший `HSET` повернув 3 — стільки **нових** полів додано. Другий — 0: поле `district` уже було, його лише змінили. `HGETALL` повертає звичайний `dict`, але зверни увагу: `delivered` — рядок `'3'`, а не число. Hash, як і рядок, зберігає тільки текст.

### Приклад 5.2. `HINCRBY`: атомарний `+=` для поля

**Задача:** кур'єр доставив замовлення — збільшити `delivered`. У словнику це `card["delivered"] += 1`; у Redis — `HINCRBY`, який, як `INCR`, не має гонитви і повертає вже число.

In [ ]:
card["delivered"] += 1
print("dict:   ", card["delivered"])
print("HINCRBY:", r.hincrby("l31:courier:1", "delivered", 1))
print("HINCRBYFLOAT виторгу:", r.hincrbyfloat("l31:courier:1", "revenue", 540.0))
print(r.hgetall("l31:courier:1"))

`HINCRBY` повертає `int`, а `HINCRBYFLOAT` — `float`; поля `revenue` не було — його створено з 0. А `HGETALL` знову віддає все рядками.

### Приклад 5.3. `HEXISTS`, `HDEL`, `HKEYS`, `HLEN`, `HMGET`

In [ ]:
print("є phone?    ", r.hexists("l31:courier:1", "phone"))
print("HGET phone: ", r.hget("l31:courier:1", "phone"))
print("HMGET:      ", r.hmget("l31:courier:1", ["name", "phone", "district"]))
print("HDEL:       ", r.hdel("l31:courier:1", "revenue"))
print("HKEYS:      ", r.hkeys("l31:courier:1"))
print("HLEN:       ", r.hlen("l31:courier:1"))

Немає поля — `None`, як `card.get("phone")`, а не `KeyError`. `HMGET` бере кілька полів одним запитом; на місці відсутнього — `None`.

### Приклад 5.4. `save_courier` і `mark_delivered`

**Задача:** функції для картки кур'єра `l31:courier:<id>`: створити з полями `name`, `district`, `delivered` = 0 і атомарно додати доставку.

Чому не зберегти картку одним JSON-рядком через `SET`? Тоді, щоб змінити одне поле, треба прочитати весь JSON, змінити й записати — знову «прочитати-змінити-записати» з гонитвою. Hash змінює одне поле однією командою.

In [ ]:
def save_courier(courier_id, name, district):
    r.hset(f"l31:courier:{courier_id}", mapping={"name": name, "district": district, "delivered": 0})


def mark_delivered(courier_id):
    return r.hincrby(f"l31:courier:{courier_id}", "delivered", 1)

In [ ]:
r.delete("l31:courier:7")
save_courier(7, "Тарас", "Центр")
print(mark_delivered(7), mark_delivered(7))
print(r.hgetall("l31:courier:7"))

assert r.hgetall("l31:courier:7") == {"name": "Тарас", "district": "Центр", "delivered": "2"}

---
## 6. Set: унікальні клієнти

**🔮 Прогноз:** `SADD day "Анна" "Богдан" "Анна"` — що поверне команда?

<details>
<summary>Відповідь</summary>

`2`: кількість **нових** елементів. Друга «Анна» вже є в множині.

</details>

### Приклад 6.1. `SADD`, `SCARD`, `SISMEMBER` і `set`

In [ ]:
MONDAY = ["Анна", "Богдан", "Віра", "Анна"]
TUESDAY = ["Анна", "Галина"]

In [ ]:
r.delete("l31:customers:mon", "l31:customers:tue")
print("SADD пн:", r.sadd("l31:customers:mon", *MONDAY))
print("SADD вт:", r.sadd("l31:customers:tue", *TUESDAY))
print("SCARD пн:", r.scard("l31:customers:mon"), "| len(set):", len(set(MONDAY)))
print("Галина в пн?", r.sismember("l31:customers:mon", "Галина"), "| in:", "Галина" in set(MONDAY))
print("SMEMBERS пн:", sorted(r.smembers("l31:customers:mon")))

`SADD` повернув 3, а не 4: дублікат «Анна» не додався. `SMEMBERS` віддає Python-`set`, порядок у ньому не гарантовано — тому `sorted`.

### Приклад 6.2. Перетин, об'єднання, різниця і `&`, `|`, `-`

In [ ]:
mon, tue = set(MONDAY), set(TUESDAY)
print("обидва дні:   SINTER", sorted(r.sinter("l31:customers:mon", "l31:customers:tue")), "| &", sorted(mon & tue))
print("хоч раз:      SUNION", sorted(r.sunion("l31:customers:mon", "l31:customers:tue")), "| |", sorted(mon | tue))
print("лише в пн:    SDIFF ", sorted(r.sdiff("l31:customers:mon", "l31:customers:tue")), "| -", sorted(mon - tue))

Ті самі операції, що з множинами в уроці 5, — тільки рахує їх сервер, і дані можуть дописувати різні програми протягом дня.

### Приклад 6.3. Постійні клієнти

`record_customers(day, names)` додає імена у `l31:customers:<day>` і повертає, скільки **унікальних** клієнтів тепер за цей день; `loyal(day1, day2)` — **відсортований** список тих, хто замовляв в обидва дні.

In [ ]:
def record_customers(day, names):
    r.sadd(f"l31:customers:{day}", *names)
    return r.scard(f"l31:customers:{day}")


def loyal(day1, day2):
    return sorted(r.sinter(f"l31:customers:{day1}", f"l31:customers:{day2}"))

In [ ]:
r.delete("l31:customers:wed", "l31:customers:thu")
print(record_customers("wed", ["Анна", "Богдан", "Віра", "Анна"]))
print(record_customers("thu", ["Анна", "Галина", "Віра"]))
print(record_customers("thu", ["Галина"]))
print(loyal("wed", "thu"))

assert loyal("wed", "thu") == ["Анна", "Віра"]

Повторна «Галина» в четвер не змінила кількість — 3. Постійні клієнти — Анна й Віра.

---
## 7. Sorted set: рейтинг і черга за дедлайном

**Відсортована множина** — множина, де в кожного елемента є **бал** (score), і Redis тримає елементи впорядкованими за ним. Кожна зміна — `O(log n)`, як у купі з уроку 29.

### Приклад 7.1. `ZADD`, `ZRANGE`, `ZREVRANGE` і `sorted`

**Задача:** рейтинг кур'єрів за виторгом з уроку 30.

In [ ]:
REVENUE = {"Оксана": 2130, "Тарас": 1570, "Ігор": 1040, "Марія": 0}

In [ ]:
r.delete("l31:rating")
print("ZADD:", r.zadd("l31:rating", REVENUE))
print("ZRANGE (зростання): ", r.zrange("l31:rating", 0, -1))
print("ZREVRANGE топ-3:    ", r.zrevrange("l31:rating", 0, 2, withscores=True))
print("sorted у Python:    ", sorted(REVENUE.items(), key=lambda item: item[1], reverse=True)[:3])

`ZADD` повертає кількість нових елементів (4). `ZRANGE` — від меншого балу до більшого, `ZREVRANGE` — навпаки; `withscores=True` додає бали, і redis-py повертає їх як `float`. У Python ми сортуємо весь словник щоразу (`O(n log n)`), а Redis тримає порядок постійно — топ-3 береться одразу.

### Приклад 7.2. `ZINCRBY`, `ZSCORE`, `ZRANK`, `ZREVRANK`

In [ ]:
print("ZINCRBY Марія +390:", r.zincrby("l31:rating", 390, "Марія"))
print("ZSCORE Тарас:      ", r.zscore("l31:rating", "Тарас"))
print("ZRANK Марія (з 0, за зростанням):", r.zrank("l31:rating", "Марія"))
print("ZREVRANK Оксана (з 0, за спаданням):", r.zrevrank("l31:rating", "Оксана"))
print("ZSCORE невідомого:", r.zscore("l31:rating", "Петро"), "| ZRANK:", r.zrank("l31:rating", "Петро"))

`ZINCRBY` додає до балу й повертає новий бал (390.0). Місця рахуються **з нуля**: `ZRANK` — від найменшого, `ZREVRANK` — від найбільшого, тож Оксана — 0, тобто перша. Для людського «першого місця» додають 1. Невідомий кур'єр — `None` в обох командах.

### Приклад 7.3. `ZRANGEBYSCORE`: усі з виторгом від 1000 до 2000

In [ ]:
print(r.zrangebyscore("l31:rating", 1000, 2000, withscores=True))
print("скільки таких:", r.zcount("l31:rating", 1000, 2000))
print("від 1500 і вище:", r.zrangebyscore("l31:rating", 1500, "+inf"))

Вибірка за діапазоном балу — без перебору всіх елементів; `+inf` — «без верхньої межі».

### Приклад 7.4. Черга за дедлайном: `ZPOPMIN` і `MinHeap` з уроку 29

**Задача:** кур'єр бере замовлення з **найближчим** дедлайном. В уроці 29 це робила купа `MinHeap`; у стандартній бібліотеці той самий алгоритм — `heapq`. У Redis бал — дедлайн (хвилини від початку зміни), а `ZPOPMIN` знімає елемент з найменшим балом.

In [ ]:
DEADLINES = {"#201 Поділ": 45, "#202 Оболонь": 20, "#203 Центр": 30, "#204 Печерськ": 60}

In [ ]:
heap = [(minutes, order) for order, minutes in DEADLINES.items()]
heapq.heapify(heap)
print("heapq:  ", [heapq.heappop(heap)[1] for _ in range(2)])

r.delete("l31:deadlines")
r.zadd("l31:deadlines", DEADLINES)
print("ZPOPMIN:", r.zpopmin("l31:deadlines", 2))
print("лишилось:", r.zrange("l31:deadlines", 0, -1, withscores=True))

Обидві черги віддали #202 (20 хв) і #203 (30 хв). `ZPOPMIN` повертає пари `(елемент, бал)` і **знімає** їх атомарно: два кур'єри ніколи не отримають одне замовлення.

### Задача 1. Рейтинг кур'єрів за тиждень

Напиши три функції на відсортованій множині `l31:rating:week`:

- `add_delivery(courier, amount)` — додати суму доставки до балу кур'єра;
- `top(n)` — список `(кур'єр, сума)` від найбільшої суми (суми — `float`, як їх повертає redis-py);
- `place(courier)` — місце кур'єра в рейтингу **з одиниці** (перший — 1).

Підказка: приклади 7.1–7.2. Перевірка — у наступній клітинці.

In [ ]:
def add_delivery(courier, amount):
    # BEGIN SOLUTION
    r.zincrby("l31:rating:week", amount, courier)
    # END SOLUTION


def top(n):
    # BEGIN SOLUTION
    return r.zrevrange("l31:rating:week", 0, n - 1, withscores=True)
    # END SOLUTION


def place(courier):
    # BEGIN SOLUTION
    return r.zrevrank("l31:rating:week", courier) + 1
    # END SOLUTION

In [ ]:
r.delete("l31:rating:week")
for courier, amount in [("Оксана", 540), ("Тарас", 1250), ("Оксана", 980), ("Ігор", 760)]:
    add_delivery(courier, amount)
print(top(3))
assert top(2) == [("Оксана", 1520.0), ("Тарас", 1250.0)], "top(2) має бути [('Оксана', 1520.0), ('Тарас', 1250.0)]: суми додаються, порядок — від більшої"
assert place("Ігор") == 3, "місце рахується з одиниці: Ігор — третій"
add_delivery("Тарас", 300)
assert place("Тарас") == 1, "після +300 Тарас має 1550 і стає першим"
print("✅ Задача 1 розв'язана")

---
## 8. redis-py і `pipeline`

### Приклад 8.1. Команди — методи

Зведемо разом те, що вже бачили: кожна команда Redis — метод клієнта, і redis-py перетворює відповіді на звичні типи Python.

In [ ]:
print(r.ping())                                     # PING      → bool
print(r.set("l31:courier:2:name", "Тарас"))         # SET       → bool
print(r.get("l31:courier:2:name"))                  # GET       → str | None
print(r.incr("l31:orders:today"))                   # INCR      → int
print(r.hgetall("l31:courier:1"))                   # HGETALL   → dict
print(r.zrevrange("l31:rating", 0, 2, withscores=True))   # ZREVRANGE → list[tuple]
print(r.smembers("l31:customers:tue"))              # SMEMBERS  → set

### Приклад 8.2. Pipeline: кілька команд за один раз

Кожна команда — запит мережею й очікування відповіді. `pipeline` збирає команди в буфер і відправляє їх разом; `execute()` повертає відповіді всіх команд списком — у тому самому порядку.

In [50]:
r.delete("l31:courier:2")
pipe = r.pipeline()
pipe.hset("l31:courier:2", mapping={"name": "Тарас", "district": "Центр", "delivered": 2})
pipe.zincrby("l31:rating", 450, "Тарас")
pipe.incr("l31:orders:today")
print(pipe.execute())

[3, 450.0, 1]


Три відповіді: 3 нових поля, новий бал Тараса (1570 + 450 = 2020.0) і нове значення лічильника. За замовчуванням pipeline у redis-py ще й **транзакційний** (`MULTI` / `EXEC`): команди виконаються підряд, і жоден інший клієнт не вклиниться між ними.

### Приклад 8.3. Пастка: до `execute()` відповідей ще немає

**🔮 Прогноз:** що надрукує `print(pipe.incr(...))` усередині pipeline?

<details>
<summary>Відповідь</summary>

Сам об'єкт pipeline, а не число: команда лише додана в буфер. Значення з'являться після `execute()`.

</details>

In [ ]:
pipe = r.pipeline()
print(pipe.incr("l31:orders:today"))
print(pipe.get("l31:orders:today"))
results = pipe.execute()
print(results)

Звідси правило: у pipeline не можна використати відповідь однієї команди для наступної — для цього потрібні окремі запити (або скрипти Lua, які ми не розбираємо).

### Приклад 8.4. Скільки економить pipeline: заміряємо

**Задача:** 1000 замовлень — 1000 `INCR`. Порівняємо три способи: по одній команді, pipeline без транзакції і транзакційний pipeline.

In [55]:
N = 1000


def one_by_one():
    for _ in range(N):
        r.incr("l31:bench")


def pipelined(transaction):
    pipe = r.pipeline(transaction=transaction)
    for _ in range(N):
        pipe.incr("l31:bench")
    return pipe.execute()


def measure(label, func, *args):
    r.delete("l31:bench")
    start = time.perf_counter()
    func(*args)
    elapsed = time.perf_counter() - start
    print(f"{label:<28} {elapsed * 1000:7.1f} мс   лічильник: {r.get('l31:bench')}")
    return elapsed

In [56]:
slow = measure("по одній команді", one_by_one)
fast = measure("pipeline", pipelined, False)
measure("pipeline + MULTI/EXEC", pipelined, True)
print(f"pipeline швидший у {slow / fast:.0f} разів")

по одній команді               823.9 мс   лічильник: 1000
pipeline                         8.6 мс   лічильник: 1000
pipeline + MULTI/EXEC            7.3 мс   лічильник: 1000
pipeline швидший у 96 разів


Результат однаковий — 1000, а час відрізняється в рази: 1000 запитів і 1000 очікувань відповіді проти одного пакета команд. Що далі сервер від програми (інший комп'ютер, інший дата-центр), то більша різниця: кожна поїздка мережею — це вже мілісекунди, а не мікросекунди. Транзакційний варіант трохи повільніший — Redis додатково обгортає команди в `MULTI` / `EXEC`.

---
## 9. Кеш: cache-aside

Звіт власниці з уроку 30 рахується довго. Схема **cache-aside** («кеш збоку»):

```
запит звіту ──▶ GET у Redis ──є (hit)──────────────────────────────▶ віддати
                    │
                    └──немає (miss)──▶ порахувати в PostgreSQL ──▶ SET … EX 300 ──▶ віддати
```

Замість справжнього SQL-запиту — повільна функція, яка рахує свої виклики. «База» — словник `ORDERS_DB`: виторг за районами.

### Приклад 9.1. Hit і miss з лічильниками

In [51]:
ORDERS_DB = {"Поділ": 2730.0, "Оболонь": 2010.0}
stats = {"hit": 0, "miss": 0, "db": 0}


def weekly_report_from_db():
    """Повільний «SELECT … GROUP BY» з уроку 30."""
    stats["db"] += 1
    time.sleep(0.5)
    return dict(ORDERS_DB)


def weekly_report(ttl=300):
    cached = r.get("l31:cache:weekly_report")
    if cached is not None:                              # hit
        stats["hit"] += 1
        return json.loads(cached)
    stats["miss"] += 1                                  # miss
    report = weekly_report_from_db()
    r.set("l31:cache:weekly_report", json.dumps(report, ensure_ascii=False), ex=ttl)
    return report


def timed_report(attempt, **kwargs):
    start = time.perf_counter()
    report = weekly_report(**kwargs)
    print(attempt, report, f"{time.perf_counter() - start:.2f} с", stats)

In [52]:
r.delete("l31:cache:weekly_report")
stats.update(hit=0, miss=0, db=0)
for attempt in range(1, 4):
    timed_report(attempt)
print("TTL кешу:", r.ttl("l31:cache:weekly_report"))
print("у Redis лежить:", r.get("l31:cache:weekly_report"))

1 {'Поділ': 2730.0, 'Оболонь': 2010.0} 0.51 с {'hit': 0, 'miss': 1, 'db': 1}
2 {'Поділ': 2730.0, 'Оболонь': 2010.0} 0.00 с {'hit': 1, 'miss': 1, 'db': 1}
3 {'Поділ': 2730.0, 'Оболонь': 2010.0} 0.00 с {'hit': 2, 'miss': 1, 'db': 1}
TTL кешу: 300
у Redis лежить: {"Поділ": 2730.0, "Оболонь": 2010.0}


Перше звернення — miss і пів секунди; наступні — hit і практично миттєво, а база працювала один раз. Redis зберігає лише рядки, тому словник звіту перетворюємо на JSON (урок 15) і назад; `ensure_ascii=False` залишає кирилицю читабельною.

### Приклад 9.2. Кеш застаріває — і як його інвалідувати

**Задача:** на Подолі нове замовлення на 500 грн. Що покаже звіт?

In [53]:
def add_order(district, amount):
    """Записати замовлення в «базу». Кеш не чіпаємо — подивимось, що буде."""
    ORDERS_DB[district] += amount


def add_order_and_invalidate(district, amount):
    """Записати замовлення і видалити застарілий кеш."""
    ORDERS_DB[district] += amount
    r.delete("l31:cache:weekly_report")

In [54]:
add_order("Поділ", 500)
print("у базі:    ", ORDERS_DB)
timed_report("без DEL:")

add_order_and_invalidate("Поділ", 500)
print("у базі:    ", ORDERS_DB)
timed_report("після DEL:")

у базі:     {'Поділ': 3230.0, 'Оболонь': 2010.0}
без DEL: {'Поділ': 2730.0, 'Оболонь': 2010.0} 0.00 с {'hit': 3, 'miss': 1, 'db': 1}
у базі:     {'Поділ': 3730.0, 'Оболонь': 2010.0}
після DEL: {'Поділ': 3730.0, 'Оболонь': 2010.0} 0.50 с {'hit': 3, 'miss': 2, 'db': 2}


Після першого замовлення звіт показав **старий** виторг Подолу (hit): кеш — копія, і вона не знає, що база змінилась. Після `DEL` наступний запит став miss, сходив у базу й порахував свіжий звіт — з обома замовленнями.

**Правило:** кеш **завжди** має TTL, а коли дані змінюються — старий запис видаляють. Скільки секунд «застарілості» допустимо — бізнес-рішення: для звіту за тиждень 5 хвилин — нормально, для балансу рахунку — ні.

### Приклад 9.3. Кеш, що закінчився сам

**Задача:** ставимо TTL 2 секунди й чекаємо 2,5 — ніхто нічого не видаляє.

In [57]:
r.delete("l31:cache:weekly_report")
timed_report("1:", ttl=2)
timed_report("2:", ttl=2)
time.sleep(2.5)
timed_report("3 (через 2,5 с):", ttl=2)

1: {'Поділ': 3730.0, 'Оболонь': 2010.0} 0.51 с {'hit': 3, 'miss': 3, 'db': 3}
2: {'Поділ': 3730.0, 'Оболонь': 2010.0} 0.00 с {'hit': 4, 'miss': 3, 'db': 3}
3 (через 2,5 с): {'Поділ': 3730.0, 'Оболонь': 2010.0} 0.50 с {'hit': 4, 'miss': 4, 'db': 4}


Третій запит знову miss: TTL минув, Redis видалив ключ, і звіт порахувався заново. TTL — «страховка»: навіть якщо ми десь забули зробити `DEL`, застарілі дані проживуть не довше за TTL.

### Задача 2. Кеш звіту по району з інвалідацією

`slow_district_report(district)` — «повільний запит до PostgreSQL» (записує свої виклики в `report_calls`). Напиши:

- `district_report(district)` — cache-aside з ключем `l31:cache:report:<district>` і TTL **120 секунд**; словник зберігай як JSON; на кожен hit збільшуй `cache_stats["hit"]`, на кожен miss — `cache_stats["miss"]`;
- `invalidate_district(district)` — видалити кеш цього району.

Підказка: приклади 9.1–9.2. Перевірка — у наступній клітинці.

In [ ]:
DISTRICT_REVENUE = {"Поділ": 2730.0, "Оболонь": 2010.0}
report_calls = []
cache_stats = {"hit": 0, "miss": 0}


def slow_district_report(district):
    report_calls.append(district)
    time.sleep(0.2)
    return {"district": district, "revenue": DISTRICT_REVENUE.get(district, 0.0)}

In [ ]:
def district_report(district):
    # BEGIN SOLUTION
    key = f"l31:cache:report:{district}"
    cached = r.get(key)
    if cached is not None:
        cache_stats["hit"] += 1
        return json.loads(cached)
    cache_stats["miss"] += 1
    report = slow_district_report(district)
    r.set(key, json.dumps(report, ensure_ascii=False), ex=120)
    return report
    # END SOLUTION


def invalidate_district(district):
    # BEGIN SOLUTION
    r.delete(f"l31:cache:report:{district}")
    # END SOLUTION

In [ ]:
r.delete("l31:cache:report:Поділ", "l31:cache:report:Оболонь")
report_calls.clear()
cache_stats.update(hit=0, miss=0)

for district in ["Поділ", "Поділ", "Оболонь", "Поділ"]:
    print(district_report(district))
assert report_calls == ["Поділ", "Оболонь"], f"база мала працювати двічі (по разу на район), а викликів: {report_calls}"
assert cache_stats == {"hit": 2, "miss": 2}, f"очікували 2 hit і 2 miss, маємо {cache_stats}"
assert 0 < r.ttl("l31:cache:report:Поділ") <= 120, "у кешу має бути TTL 120 секунд"

DISTRICT_REVENUE["Поділ"] += 500
assert district_report("Поділ")["revenue"] == 2730.0      # ще стара копія
invalidate_district("Поділ")
assert district_report("Поділ")["revenue"] == 3230.0, "після invalidate_district звіт має порахуватися заново"
assert district_report("Оболонь") == {"district": "Оболонь", "revenue": 2010.0}
DISTRICT_REVENUE["Поділ"] -= 500
print("база працювала", len(report_calls), "рази на", sum(cache_stats.values()), "запитів")
print("✅ Задача 2 розв'язана")

---
## 10. Черга задач: producer і consumer

Приймання замовлення не повинно чекати SMS-шлюз. Сервіс-**producer** кладе задачу в список Redis і одразу відповідає клієнтові; окремий процес-**consumer** (воркер) забирає задачі й відправляє SMS.

```
 сервіс замовлень ──RPUSH──▶ [ l31:queue:sms ] ──BLPOP──▶ воркер 1 ──▶ SMS-шлюз
     (producer)                (список у Redis) ──BLPOP──▶ воркер 2 ──▶ SMS-шлюз
```

### Приклад 10.1. Producer і consumer

In [ ]:
def send_sms_later(phone, text):
    """Producer: покласти задачу в чергу й повернутися одразу."""
    r.rpush("l31:queue:sms", json.dumps({"phone": phone, "text": text}, ensure_ascii=False))


def sms_worker(max_tasks, timeout=1):
    """Consumer: брати задачі з голови черги; чекати, якщо черга порожня."""
    for _ in range(max_tasks):
        item = r.blpop("l31:queue:sms", timeout=timeout)
        if item is None:
            print(f"черга порожня — воркер чекав {timeout} с і зупинився")
            return
        _queue_name, payload = item
        task = json.loads(payload)
        print("відправлено:", task["phone"], "—", task["text"])

In [ ]:
r.delete("l31:queue:sms")
send_sms_later("+380501112233", "Замовлення #101 прийнято")
send_sms_later("+380672223344", "Кур'єр уже їде")
print("у черзі:", r.llen("l31:queue:sms"))
sms_worker(max_tasks=3)

`BLPOP` — «блокуючий» `LPOP`: повертає пару `(назва черги, елемент)`, а якщо черга порожня — **чекає**, доки задача з'явиться, або повертає `None`, коли мине `timeout`. Третя спроба чекала секунду й не дочекалась.

### Приклад 10.2. Воркер чекає задачу

**Задача:** переконатися, що воркер справді спить і прокидається, щойно задача з'являється. Запустимо воркера в окремому потоці (в реальному житті це окрема програма), а задачу покладемо через секунду.

In [ ]:
def waiting_worker(log):
    start = time.perf_counter()
    _queue, payload = r.blpop("l31:queue:sms", timeout=5)
    log.append((round(time.perf_counter() - start, 1), json.loads(payload)["text"]))

In [ ]:
r.delete("l31:queue:sms")
log = []
worker = threading.Thread(target=waiting_worker, args=(log,))
worker.start()

time.sleep(1)                           # клієнт оформлює замовлення
send_sms_later("+380931234567", "Замовлення #105 прийнято")
worker.join()
print("воркер чекав і отримав:", log)

Воркер чекав близько секунди — рівно стільки, скільки черга була порожня, — і отримав задачу одразу після `RPUSH`. Ніякого циклу «перевіряй щосекунди»: Redis сам будить того, хто чекає.

### Приклад 10.3. Два воркери — кожна задача рівно одному

In [ ]:
def worker_loop(name, done):
    """Брати задачі, доки черга не стане порожньою на 0,5 с."""
    while True:
        item = r.blpop("l31:queue:sms", timeout=0.5)
        if item is None:
            return
        done.append((name, json.loads(item[1])["text"]))
        time.sleep(0.05)                 # «відправка SMS»

In [ ]:
r.delete("l31:queue:sms")
for number in range(301, 309):
    send_sms_later("+380500000000", f"замовлення #{number}")

done = []
workers = [threading.Thread(target=worker_loop, args=(name, done)) for name in ["воркер-1", "воркер-2"]]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()

for name in ["воркер-1", "воркер-2"]:
    print(name, [text for who, text in done if who == name])
texts = [text for _who, text in done]
assert sorted(texts) == [f"замовлення #{number}" for number in range(301, 309)]
assert len(texts) == len(set(texts))

Вісім задач розійшлися між двома воркерами, жодна не загубилась і жодна не дісталась обом: `BLPOP` знімає елемент атомарно. Більше воркерів — більше SMS за секунду. Так працює Celery (уроки 49+): Redis — брокер, воркери — окремі процеси.

---
## 11. Pub/Sub: сповіщення «всім, хто слухає»

Черга — «одна задача одному воркеру, і вона чекає». **Pub/Sub** — навпаки: повідомлення отримують **усі** підписники каналу, а хто не слухав — не отримає нічого. Канали Pub/Sub спільні для всього сервера (номер бази на них не впливає), тому й назви каналів починаємо з `l31:`.

### Приклад 11.1. Підписатися й опублікувати

In [ ]:
listener = r.pubsub()
listener.subscribe("l31:dispatch:events")
print("підтвердження:", listener.get_message(timeout=1))

print("PUBLISH отримали:", r.publish("l31:dispatch:events", "нове термінове замовлення #104"))
message = listener.get_message(timeout=1)
print(message["channel"], "→", message["data"])
listener.close()

Перше повідомлення — службове підтвердження підписки (`type: subscribe`). `PUBLISH` повертає, **скільки підписників** отримали повідомлення, — один.

### Приклад 11.2. Ніхто не слухає — повідомлення зникло

**🔮 Прогноз:** публікуємо, коли підписників немає, а потім підписуємось. Отримаємо повідомлення?

<details>
<summary>Відповідь</summary>

Ні. `PUBLISH` поверне 0, і повідомлення ніде не збережеться. А задача в черзі (list) чекала б, доки її заберуть.

</details>

Щоб далі не повторювати «підписатися й прочитати підтвердження», загорнемо це у функцію. Поки підтвердження не прийшло, підписка може ще не діяти, і повідомлення, опубліковане в цей момент, пройде повз.

In [ ]:
def subscribe(channel):
    """Підписатися на канал і дочекатися підтвердження від сервера."""
    pubsub = r.pubsub()
    pubsub.subscribe(channel)
    confirmation = pubsub.get_message(timeout=1)
    assert confirmation["type"] == "subscribe"
    return pubsub

In [ ]:
print("PUBLISH без слухачів:", r.publish("l31:dispatch:events", "замовлення #105"))

late = subscribe("l31:dispatch:events")
print("запізнілий слухач отримав:", late.get_message(timeout=1))
late.close()

r.delete("l31:queue:sms")
r.rpush("l31:queue:sms", "замовлення #105")
print("а в черзі воно лежить:", r.lrange("l31:queue:sms", 0, -1))

### Приклад 11.3. Два екрани диспетчерів отримують одне повідомлення

In [ ]:
screens = [subscribe("l31:dispatch:events") for _ in range(2)]

print("PUBLISH отримали:", r.publish("l31:dispatch:events", "кур'єр Оксана вільна"))
for number, screen in enumerate(screens, start=1):
    message = screen.get_message(timeout=1)
    print(f"екран {number}:", message["data"])
    screen.close()

Обидва екрани отримали те саме повідомлення — на відміну від черги з прикладу 10.3, де кожна задача дісталась одному воркеру.

| | Черга (list + `BLPOP`) | Pub/Sub |
|---|---|---|
| Хто отримує | один воркер | усі підписники |
| Якщо ніхто не слухає | задача чекає в Redis | повідомлення зникає |
| Для чого | задачі, які не можна загубити: SMS, листи | «живі» сповіщення: екран диспетчера, чат (урок 46) |

---
## 12. Персистентність і пам'ять

Redis тримає дані в пам'яті. Що буде після перезапуску сервера і що — коли пам'ять закінчиться? Подивимось на налаштування нашого сервера командою `CONFIG GET`. На хмарних Redis (керованих провайдером) `CONFIG` часто заборонено — тоді клітинки надрукують «недоступно».

- **RDB** — знімок усієї пам'яті на диск за правилами `save`: «якщо за N секунд було хоча б M змін». Після падіння втрачається все, що змінилось після останнього знімка.
- **AOF** (`appendonly`) — журнал кожної команди. Втрати менші, але файл більший, а відновлення повільніше.

In [ ]:
def config_get(name):
    """CONFIG GET name → значення або «недоступно» (хмарний Redis може забороняти CONFIG)."""
    try:
        return r.config_get(name).get(name, "немає такого параметра")
    except redis.ResponseError as error:
        return f"недоступно ({error})"

### Приклад 12.1. Як наш сервер зберігає дані: `save`, `appendonly`

In [ ]:
for name in ["save", "appendonly", "appendfsync", "dir", "dbfilename"]:
    print(f"{name:<12} {config_get(name)!r}")

persistence = r.info("persistence")
print("AOF увімкнено:            ", persistence["aof_enabled"])
print("змін від останнього знімка:", persistence["rdb_changes_since_last_save"])

`save` — правила RDB-знімків парами «секунд змін»: наприклад, `3600 1 300 100 60 10000` означає «знімок, якщо за годину була хоча б 1 зміна, за 5 хвилин — 100, за хвилину — 10 000». `appendonly no` — AOF вимкнено. Отже, за замовчуванням після падіння можна втратити до кількох хвилин змін — для кешу й лічильників це нормально, для грошей — ні.

### Приклад 12.2. Знімок вручну: `BGSAVE`

`BGSAVE` робить RDB-знімок у фоні (сервер не зупиняється). Після нього лічильник «змін від останнього знімка» обнуляється, а `LASTSAVE` показує час знімка.

In [ ]:
def snapshot():
    """Зробити RDB-знімок і дочекатися його завершення."""
    before = r.lastsave()
    try:
        r.bgsave()
    except redis.ResponseError as error:
        print("BGSAVE недоступний:", error)
        return
    while r.lastsave() == before:
        time.sleep(0.2)
    print("знімок зроблено:", r.lastsave())

In [ ]:
for number in range(5):
    r.incr("l31:orders:today")
print("змін до знімка:   ", r.info("persistence")["rdb_changes_since_last_save"])
snapshot()
print("змін після знімка:", r.info("persistence")["rdb_changes_since_last_save"])

Знімок містить **усю** базу, але між знімками зміни живуть лише в пам'яті. Навіть з AOF Redis не дає гарантій PostgreSQL: немає обмежень, зовнішніх ключів, складних транзакцій. Тому гроші й замовлення — у PostgreSQL.

### Приклад 12.3. Що буде, коли закінчиться пам'ять: `maxmemory-policy`

`maxmemory` — скільки пам'яті Redis може взяти (0 — без обмеження), `maxmemory-policy` — що робити, коли її не вистачає. Змінимо політику на `allkeys-lru` і **повернемо назад**: це налаштування всього сервера, а не нашої бази.

In [ ]:
def show_memory_policy():
    print("maxmemory:       ", config_get("maxmemory"))
    print("maxmemory-policy:", config_get("maxmemory-policy"))

In [ ]:
show_memory_policy()
original_policy = config_get("maxmemory-policy")

try:
    r.config_set("maxmemory-policy", "allkeys-lru")
    print("\nпісля CONFIG SET:")
    show_memory_policy()
    r.config_set("maxmemory-policy", original_policy)
    print("\nповернули:", config_get("maxmemory-policy"))
except redis.ResponseError as error:
    print("CONFIG SET недоступний:", error)

За замовчуванням `noeviction`: коли пам'ять заповнена, **нові записи отримують помилку**, а старі дані не чіпаються. `allkeys-lru` робить з усього Redis **LRU-кеш з уроку 29**: викидає ключі, до яких найдовше не зверталися. Для сервера-кешу — саме те; для черги задач — небезпечно (зникнуть задачі). Тому кеш і черги часто тримають у **різних** екземплярах Redis. Інші політики: `volatile-lru` (викидати лише ключі з TTL), `allkeys-lfu` (ті, до яких зверталися найрідше), `volatile-ttl` (ті, що й так скоро зникнуть).

### Приклад 12.4. Скільки пам'яті займають наші ключі

`MEMORY USAGE` — скільки байтів займає ключ разом зі службовими даними; `info("memory")` — скільки пам'яті використовує весь сервер.

In [ ]:
for key in ["l31:orders:today", "l31:courier:1", "l31:rating", "l31:events"]:
    print(f"{key:<18} {r.type(key):<7} {r.memory_usage(key)} байт")
print("увесь сервер:", r.info("memory")["used_memory_human"])

Навіть лічильник з одним числом займає десятки байтів: сам ключ, службовий заголовок, час життя. Мільйон таких ключів — це вже десятки мегабайтів **оперативної** пам'яті. Тому в Redis тримають те, що справді потрібно швидко, і майже завжди — з TTL.

---
## 13. Коли Redis не потрібен

Redis швидкий — але «швидкий» порівняно з PostgreSQL, а не з пам'яттю твоєї програми. Кожен запит — поїздка мережею.

### Приклад 13.1. Словник Python проти Redis

In [ ]:
MENU_PRICES = {f"страва {number}": 100 + number for number in range(100)}


def read_dict(times):
    for number in range(times):
        MENU_PRICES[f"страва {number % 100}"]


def read_redis(times):
    for number in range(times):
        r.hget("l31:menu", f"страва {number % 100}")

In [ ]:
r.delete("l31:menu")
r.hset("l31:menu", mapping=MENU_PRICES)

for label, func in [("dict", read_dict), ("Redis", read_redis)]:
    start = time.perf_counter()
    func(2000)
    per_call = (time.perf_counter() - start) / 2000 * 1_000_000
    print(f"{label:<6} {per_call:8.2f} мкс на читання")

Словник у пам'яті програми — у сотні, а то й тисячі разів швидший: жодної мережі. Redis потрібен, коли дані мають бути **спільними** для кількох процесів чи серверів, переживати перезапуск програми або мати TTL — а не просто «щоб було швидко».

Коли Redis зайвий:

- один сервер і дані, які можна тримати в пам'яті процесу, — `dict` чи `functools.lru_cache`;
- PostgreSQL справляється: один звіт на годину, сотня замовлень на день;
- дані, які не можна втратити, — їхнє місце в PostgreSQL.

Кожен додатковий сервер — ще одна річ, яка може впасти, і питання «а чи свіжий кеш?». Спершу **виміряй** (урок 9), потім додавай кеш.

```
нові дані ──▶ втрата — катастрофа? (гроші, замовлення) ──так──▶ PostgreSQL
                   │ ні
                   ▼
              можна відновити або живе недовго?
                   ├── копія з бази ──────▶ Redis-кеш з TTL
                   ├── лічильник, сесія ──▶ Redis з TTL
                   └── задача у фоні ─────▶ Redis-черга (list / Celery)
```

---
## 14. Як Redis говорить мережею: RESP

Redis спілкується через TCP простим текстовим протоколом **RESP**. Команда — масив рядків: `*<кількість>`, потім кожен рядок як `$<довжина в байтах>` і сам рядок; розділювач — `\r\n`. Саме це робить redis-py за тебе. Сокети й TCP докладно — в уроці 32.

### Приклад 14.1. Збираємо команду в байти

In [ ]:
def encode_command(*args):
    """Команда Redis → байти RESP: масив рядків-«bulk string»."""
    parts = [f"*{len(args)}\r\n".encode()]
    for arg in args:
        data = str(arg).encode("utf-8")
        parts.append(f"${len(data)}\r\n".encode() + data + b"\r\n")
    return b"".join(parts)

In [ ]:
print(encode_command("PING"))
print(encode_command("GET", "l31:orders:today"))
print(encode_command("SET", "l31:courier:9:name", "Ліна"))

`*1` — масив з одного елемента, `$4` — рядок довжиною 4 байти. Зверни увагу на «Ліна»: `$8`, бо кожна кирилична літера в UTF-8 — 2 байти. Довжина — у **байтах**, а не в символах.

### Приклад 14.2. Розмова з Redis звичайним сокетом

Візьмемо адресу з нашого клієнта `r`, відкриємо TCP-з'єднання й надішлемо команди без жодної бібліотеки.

In [ ]:
def resp_call(sock, *args):
    """Надіслати одну команду й повернути сиру відповідь сервера."""
    sock.sendall(encode_command(*args))
    time.sleep(0.05)
    return sock.recv(4096)


settings = r.connection_pool.connection_kwargs
HOST, PORT, DB = settings.get("host", "localhost"), settings.get("port", 6379), settings.get("db", 0)

In [ ]:
with socket.create_connection((HOST, PORT)) as sock:
    if settings.get("password"):
        print(resp_call(sock, "AUTH", *([settings["username"]] if settings.get("username") else []), settings["password"]))
    print("SELECT   ", resp_call(sock, "SELECT", DB))
    print("PING     ", resp_call(sock, "PING"))
    print("SET      ", resp_call(sock, "SET", "l31:raw:counter", 41))
    print("INCR     ", resp_call(sock, "INCR", "l31:raw:counter"))
    print("GET      ", resp_call(sock, "GET", "l31:raw:counter"))
    print("GET nil  ", resp_call(sock, "GET", "l31:raw:missing"))
    print("RPUSH    ", resp_call(sock, "RPUSH", "l31:raw:list", "a", "b"))
    print("LRANGE   ", resp_call(sock, "LRANGE", "l31:raw:list", 0, -1))
    print("помилка  ", resp_call(sock, "INCR", "l31:raw:list"))

print("redis-py бачить те саме:", r.get("l31:raw:counter"))

Перший байт відповіді каже, що це:

| Початок | Тип | Приклад з виводу |
|---|---|---|
| `+` | простий рядок | `+OK`, `+PONG` |
| `:` | ціле число | `:42` — результат `INCR` |
| `$` | рядок заданої довжини | `$2\r\n42` — `GET`; `$-1` — nil, тобто `None` |
| `*` | масив | `*2\r\n$1\r\na\r\n$1\r\nb` — `LRANGE` |
| `-` | помилка | `-WRONGTYPE …` — `INCR` над списком |

redis-py робить те саме: кодує аргументи, надсилає байти й розбирає відповідь у `True`, `int`, `str`, `None`, `list` або виняток `ResponseError`.

---
## 15. Практика: rate limit

### Приклад 15.1. Не більше трьох замовлень за хвилину

Хтось скриптом надсилає сотні фейкових замовлень. Обмежимо: **не більше 3 замовлень за хвилину з одного номера телефону**. Ключ — номер плюс поточна хвилина; значення — лічильник; TTL — 60 секунд.

In [ ]:
def allow_order(phone, minute):
    key = f"l31:rate:{phone}:{minute}"
    count = r.incr(key)          # атомарно: +1 і повернути нове значення
    if count == 1:
        r.expire(key, 60)        # перше замовлення хвилини — ключ зникне сам
    return count <= 3

In [ ]:
for key in r.scan_iter("l31:rate:*"):
    r.delete(key)

for attempt in range(1, 6):
    print(attempt, allow_order("+380501112233", "2026-09-26T12:05"))
print("інша хвилина:", allow_order("+380501112233", "2026-09-26T12:06"))
print("інший номер:", allow_order("+380672223344", "2026-09-26T12:05"))
print("TTL:", r.ttl("l31:rate:+380501112233:2026-09-26T12:05"))

- Хвилина — частина ключа: нова хвилина — новий лічильник з нуля. Тут вона передається параметром, щоб результат можна було перевірити.
- Старі лічильники прибере TTL — пам'ять не забивається вчорашніми хвилинами.

### Приклад 15.2. Десять запитів одночасно

**🔮 Прогноз:** 10 потоків одночасно викликають `allow_order` для одного номера в одну хвилину. Скільки отримають `True`?

<details>
<summary>Відповідь</summary>

Рівно 3. `INCR` атомарний: кожен потік отримує своє число від 1 до 10, і лише ті, кому дісталися 1, 2, 3, проходять. Якби ми рахували через `GET` + `SET`, «пролізти» могло б більше.

</details>

In [ ]:
def burst(phone, minute, requests=10):
    """`requests` потоків одночасно перевіряють ліміт; повертає список відповідей."""
    answers = []
    barrier = threading.Barrier(requests)        # стартуємо всі разом

    def client():
        barrier.wait()
        answers.append(allow_order(phone, minute))

    threads = [threading.Thread(target=client) for _ in range(requests)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return answers

In [ ]:
answers = burst("+380990001122", "2026-09-26T12:07")
print(answers)
print("пропущено:", answers.count(True), "з", len(answers))
assert answers.count(True) == 3

Порядок `True` і `False` у списку щоразу інший — потоки завершуються в різному порядку, — але пропущено завжди рівно 3.

### Приклад 15.3. Справжня поточна хвилина

У сервісі хвилину беруть з годинника (урок 13): `datetime.now().strftime("%Y-%m-%dT%H:%M")`. Два виклики підряд майже завжди потрапляють в одну хвилину.

In [ ]:
from datetime import datetime


def allow_order_now(phone):
    return allow_order(phone, datetime.now().strftime("%Y-%m-%dT%H:%M"))

In [ ]:
print([allow_order_now("+380631110000") for _ in range(4)])
print(sorted(r.scan_iter("l31:rate:+380631110000:*")))

Четверте замовлення відхилено (якщо клітинка не потрапила рівно на зміну хвилини — тоді лічильник почався б заново). Ключ містить поточну хвилину — через 60 секунд він зникне.

### Задача 3. Ліміт для різних дій

Узагальни `allow_order` до `allow(action, user, minute, limit)`: не більше `limit` дій за хвилину. Ключ — `l31:rate:<action>:<user>:<minute>` (дія в ключі, щоб лічильники різних дій не змішувались), `INCR`, TTL 60 секунд на першій дії. Повертає `True`, якщо дію дозволено.

Перевірка — у наступній клітинці.

In [ ]:
def allow(action, user, minute, limit):
    # BEGIN SOLUTION
    key = f"l31:rate:{action}:{user}:{minute}"
    count = r.incr(key)
    if count == 1:
        r.expire(key, 60)
    return count <= limit
    # END SOLUTION

In [ ]:
for key in r.scan_iter("l31:rate:*:+380501112233:*"):
    r.delete(key)

promo = [allow("promo", "+380501112233", "12:05", 5) for _ in range(6)]
print(promo)
assert promo == [True] * 5 + [False], f"6 перевірок промокоду з лімітом 5: очікували True ×5, потім False, маємо {promo}"
assert allow("order", "+380501112233", "12:05", 3) is True, "інша дія — свій лічильник"
assert allow("promo", "+380501112233", "12:06", 5) is True, "нова хвилина — лічильник з нуля"
assert 0 < r.ttl("l31:rate:promo:+380501112233:12:05") <= 60, "у ключа має бути TTL 60 секунд"
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Що класти в Redis, а що — лише в PostgreSQL?
2. Чому `INCR` безпечний з кількох серверів, а `GET` + `SET` — ні?
3. Що повернуть `TTL` для ключа без часу життя і для ключа, якого немає? Що станеться з TTL після звичайного `SET`?
4. Яку структуру Redis взяти для: черги SMS; рейтингу кур'єрів; картки кур'єра; унікальних клієнтів за день?
5. Опиши cache-aside. Чому кеш без TTL — погана ідея і навіщо `DEL` після зміни даних?
6. Чому pipeline швидший за окремі команди?
7. Чим черга на list відрізняється від Pub/Sub?
8. Що станеться із заповненим Redis за `noeviction` і за `allkeys-lru`?

<details>
<summary>Відповіді</summary>

1. Redis — те, що можна відновити або що живе недовго: кеш, лічильники, черги, сесії. PostgreSQL — джерело правди: замовлення, гроші.
2. Redis виконує команди по одній; `INCR` — одна команда. Між `GET` і `SET` встигають команди інших клієнтів (приклад 2.3).
3. `-1` і `-2`. Звичайний `SET` стирає TTL — ключ стає вічним; зберегти TTL — `keepttl=True`.
4. List; sorted set; hash; set.
5. Шукаємо в Redis; немає — рахуємо в базі й кладемо в Redis з TTL. Без TTL копія житиме вічно й показуватиме застарілі дані; `DEL` прибирає копію одразу, щоб наступний запит порахував свіже.
6. Команди йдуть одним пакетом: одна поїздка мережею й одне очікування замість тисячі.
7. У черзі кожну задачу бере один воркер, і вона чекає в Redis, доки її заберуть. Pub/Sub розсилає повідомлення всім підписникам одразу й ніде його не зберігає.
8. `noeviction` — нові записи отримають помилку; `allkeys-lru` — Redis викине ключі, до яких найдовше не зверталися.

</details>

In [ ]:
print("прибрано ключів уроку:", cleanup())
r.close()

## Далі

- Книга: [Урок 31](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m3/lesson_31/) — схеми, `redis-cli`, архітектура «що класти в Redis».
- Документація: [Redis data types](https://redis.io/docs/latest/develop/data-types/), [Commands](https://redis.io/docs/latest/commands/), [Persistence](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/), [Key eviction](https://redis.io/docs/latest/develop/reference/eviction/), [RESP](https://redis.io/docs/latest/develop/reference/protocol-spec/), [redis-py](https://redis.readthedocs.io/en/stable/).
- **Урок 32** — HTTP: `requests`, `httpx`, `aiohttp`.
- Redis у веб-застосунку — урок 40 «Middlewares і кешування»; Pub/Sub для чату — урок 46; Celery з Redis-брокером — уроки 49–50.